<a href="https://colab.research.google.com/github/shashi3876/Causal_AI/blob/main/SMD.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

# Load data and EDA

Here, we load the Lalonde dataset using a URL. (The Lalonde dataset was previously available in the causaldata and econml packages, but I was unable to find it recently. Thus, I am relying on a publicly available URL of the dataset on GitHub.)

The Lalonde dataset is a widely used dataset in causal inference and econometrics, originally introduced by Robert Lalonde (1986) in his study on evaluating the effectiveness of job training programs. It contains real-world observational data, where treatment and control groups are compared to assess the causal impact of the National Supported Work (NSW) program on participants' earnings. The dataset includes variables such as age, education, race, marital status, prior earnings, and post-intervention earnings.

In [ ]:
# URL of the dataset
url = 'https://raw.githubusercontent.com/robjellis/lalonde/master/lalonde_data.csv'

lalonde_df = pd.read_csv(url)

lalonde_df.head()

,ID,treat,age,educ,black,hispan,married,nodegree,re74,re75,re78
0,NSW1,1,37,11,1,0,1,1,0.0,0.0,9930.0460
1,NSW2,1,22,9,0,1,0,1,0.0,0.0,3595.8940
2,NSW3,1,30,12,1,0,0,0,0.0,0.0,24909.4500
3,NSW4,1,27,11,1,0,0,1,0.0,0.0,7506.1460
4,NSW5,1,33,8,1,0,0,1,0.0,0.0,289.7899


In [ ]:
lalonde_df.describe()

,treat,age,educ,black,hispan,married,nodegree,re74,re75,re78
count,614.000000,614.000000,614.000000,614.000000,614.000000,614.000000,614.000000,614.000000,614.000000,614.000000
mean,0.301303,27.363192,10.268730,0.395765,0.117264,0.415309,0.630293,4557.546569,2184.938207,6792.834483
std,0.459198,9.881187,2.628325,0.489413,0.321997,0.493177,0.483119,6477.964479,3295.679043,7470.730792
min,0.000000,16.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,20.000000,9.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,238.283425
50%,0.000000,25.000000,11.000000,0.000000,0.000000,0.000000,1.000000,1042.330000,601.548400,4759.018500
75%,1.000000,32.000000,12.000000,1.000000,0.000000,1.000000,1.000000,7888.498250,3248.987500,10893.592500
max,1.000000,55.000000,18.000000,1.000000,1.000000,1.000000,1.000000,35040.070000,25142.240000,60307.930000


This dataset contains 11 columns, including an ID column, which serves as a unique identifier. It includes 614 subjects, indicating whether they were part of the treatment group (i.e., whether they attended the program or not).

There are 8 columns that provide additional information about each participant, including their age, income in previous years, and ethnicity. The column re78 represents their income in 1978.

The primary objective of this analysis is to determine whether attending the program (treatment = 1) has a significant impact on future earnings (re78).

In [ ]:
# prompt: Give me the code to obtaine SMD values for a given data set. Input will be the dataframe, the name of the treament column, and the names (list) of confounder variables. Assume the treatemnt is binary

import pandas as pd
import numpy as np

def calculate_smd(df, treatment_col, confounders):
    """
    Calculates standardized mean differences (SMDs) for a given dataset.

    Args:
        df: The input DataFrame.
        treatment_col: The name of the treatment column (binary).
        confounders: A list of confounder variable names.

    Returns:
        A pandas Series containing the SMDs for each confounder.
    """

    treated = df[df[treatment_col] == 1]
    control = df[df[treatment_col] == 0]
    smd_values = {}

    for confounder in confounders:
        mean_treated = np.mean(treated[confounder])
        mean_control = np.mean(control[confounder])
        sd_treated = np.std(treated[confounder])
        sd_control = np.std(control[confounder])

        #Pooled Standard Deviation
        pooled_sd = np.sqrt(((len(treated) - 1) * sd_treated**2 + (len(control) - 1) * sd_control**2) / (len(treated) + len(control) - 2))

        smd = (mean_treated - mean_control) / pooled_sd
        smd_values[confounder] = smd
        smd_results = pd.Series(smd_values)
        smd_results.index.name = 'SMD'
    return smd_results

# Example usage (assuming 'lalonde_df' is already loaded as in your provided code):
# Replace with your actual treatment column and confounders
treatment_column = 'treat'
confounder_variables = ['age', 'educ', 'black', 'hispan', 'married', 'nodegree', 're74', 're75']


smd_results = calculate_smd(lalonde_df, treatment_column, confounder_variables)
smd_results


,0
SMD,
age,-0.225401
educ,0.042082
black,1.638375
hispan,-0.258897
married,-0.688669
nodegree,0.232007
re74,-0.562107
re75,-0.286194


In [ ]:
def calculate_smd(df, treatment_col, confounders):
    """
    Calculates standardized mean differences (SMDs) for a given dataset.

    Args:
        df: The input DataFrame.
        treatment_col: The name of the treatment column (binary).
        confounders: A list of confounder variable names.

    Returns:
        A pandas Series containing the SMDs for each confounder.
    """

    treated = df[df[treatment_col] == 1]
    control = df[df[treatment_col] == 0]
    smd_values = {}

    for confounder in confounders:
        mean_treated = np.mean(treated[confounder])
        mean_control = np.mean(control[confounder])
        sd_treated = np.std(treated[confounder])
        sd_control = np.std(control[confounder])

        #Pooled Standard Deviation
        pooled_sd = np.sqrt(((len(treated) - 1) * sd_treated**2 + (len(control) - 1) * sd_control**2) / (len(treated) + len(control) - 2))

        smd = (mean_treated - mean_control) / pooled_sd
        smd_values[confounder] = smd

    smd_results = pd.Series(smd_values)
    smd_results.name = 'SMD'
    return smd_results

# Example usage:
treatment_column = 'treat'
confounder_variables = ['age', 'educ', 'black', 'hispan', 'married', 'nodegree', 're74', 're75']

smd_results = calculate_smd(lalonde_df, treatment_column, confounder_variables)
smd_results


,SMD
age,-0.225401
educ,0.042082
black,1.638375
hispan,-0.258897
married,-0.688669
nodegree,0.232007
re74,-0.562107
re75,-0.286194
